# 🎮 Steam Video Games Analysis
This project explores the video games published on Steam using PySpark and Databricks.
We aim to understand what makes a game popular and how the market has evolved over time.


In [0]:
# Spark initialization

spark  # Shows the SparkSession object
# Get the SparkContext from the session
sc = spark.sparkContext

# Print session details
print(f"Application Name: {sc.appName}")
print(f"Cluster Master: {sc.master}")


Application Name: Databricks Shell
Cluster Master: local[8]


## 📂 1. Data Loading and Initial Exploration
We load the dataset from S3 and explore its schema and structure.

In [0]:
# Load the JSON dataset from S3

df = spark.read.option("multiLine", True).json("s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json")


In [0]:
# Count the number of rows
row_count = df.count()

# Count the number of columns
column_count = len(df.columns)

print(f"The DataFrame has {row_count} rows and {column_count} columns.")


The DataFrame has 55691 rows and 2 columns.


In [0]:
# Display schema

df.printSchema()


root
 |-- data: struct (nullable = true)
 |    |-- appid: long (nullable = true)
 |    |-- categories: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- ccu: long (nullable = true)
 |    |-- developer: string (nullable = true)
 |    |-- discount: string (nullable = true)
 |    |-- genre: string (nullable = true)
 |    |-- header_image: string (nullable = true)
 |    |-- initialprice: string (nullable = true)
 |    |-- languages: string (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- negative: long (nullable = true)
 |    |-- owners: string (nullable = true)
 |    |-- platforms: struct (nullable = true)
 |    |    |-- linux: boolean (nullable = true)
 |    |    |-- mac: boolean (nullable = true)
 |    |    |-- windows: boolean (nullable = true)
 |    |-- positive: long (nullable = true)
 |    |-- price: string (nullable = true)
 |    |-- publisher: string (nullable = true)
 |    |-- release_date: string (nullable = true)
 |    |-

The dataset is a deeply nested JSON structure containing a large volume of data. Therefore, the analysis is conducted on Databricks using PySpark, which is well-suited for handling semi-structured data at scale thanks to its distributed processing capabilities.

## 📊 2. Macro-Level Analysis
Number of games by publisher, ratings, discounts, release trends, etc.

##### Which publisher has released the most games on Steam?

In [0]:
# Flatten the JSON structure to access top-level fields
df_flat = df.select("data.*")

# Group by publisher and count the number of games
publisher_counts = df_flat.groupBy("publisher").count()

# Sort in descending order to find the most prolific publisher
publisher_counts_sorted = publisher_counts.orderBy("count", ascending=False)

# Show the top 10 publishers
publisher_counts_sorted.show(10, truncate=False)


+---------------+-----+
|publisher      |count|
+---------------+-----+
|Big Fish Games |422  |
|8floor         |202  |
|SEGA           |165  |
|Strategy First |151  |
|Square Enix    |141  |
|Choice of Games|140  |
|Sekai Project  |132  |
|HH-Games       |132  |
|               |132  |
|Ubisoft        |127  |
+---------------+-----+
only showing top 10 rows



The initial analysis of the top 10 publishers revealed that one of the entries had a missing (null) value for the publisher name. This suggests that the dataset contains incomplete records and should be reviewed for data quality. To ensure accurate insights, we revised the code to exclude null publishers from the analysis. The updated top 10 publishers list now only includes valid, named entities.

In [0]:
from pyspark.sql.functions import col

# Flatten the structure
df_flat = df.select("data.*")

# Filter out null or empty publishers
df_clean_publishers = df_flat.filter(
    (col("publisher").isNotNull()) & (col("publisher") != "")
)

# Group by publisher and count games
publisher_counts = df_clean_publishers.groupBy("publisher").count()

# Sort by count descending
publisher_counts_sorted = publisher_counts.orderBy("count", ascending=False)

# Show top 10 publishers
publisher_counts_sorted.show(10, truncate=False)



+---------------+-----+
|publisher      |count|
+---------------+-----+
|Big Fish Games |422  |
|8floor         |202  |
|SEGA           |165  |
|Strategy First |151  |
|Square Enix    |141  |
|Choice of Games|140  |
|Sekai Project  |132  |
|HH-Games       |132  |
|Ubisoft        |127  |
|Laush Studio   |126  |
+---------------+-----+
only showing top 10 rows



##### What are the best rated games?

We chose to use the ratio of positive to total reviews as a more balanced indicator of a game's quality, and in case of ties, we prioritized games with a higher number of positive votes to favor those with broader player feedback.

In [0]:
from pyspark.sql.functions import col

# Flatten structure
df_flat = df.select("data.*")

# Filter games with at least 100 votes
df_filtered = df_flat.filter((col("positive") + col("negative") >= 100))

# Add positive ratio column
df_with_rating = df_filtered.withColumn(
    "positive_ratio", col("positive") / (col("positive") + col("negative"))
)

# Sort by positive_ratio (desc), then by positive count (desc)
df_best_rated = df_with_rating.select("name", "positive", "negative", "positive_ratio") \
    .orderBy(col("positive_ratio").desc(), col("positive").desc())

# Show top 10
df_best_rated.show(10, truncate=False)


+-------------------------------------------------+--------+--------+--------------+
|name                                             |positive|negative|positive_ratio|
+-------------------------------------------------+--------+--------+--------------+
|The Void Rains Upon Her Heart                    |496     |0       |1.0           |
|祈風 Inorikaze                                   |327     |0       |1.0           |
|秘封旅行 ~ Secret Sealing Travel                 |218     |0       |1.0           |
|Elasto Mania Remastered                          |190     |0       |1.0           |
|Freshly Frosted                                  |157     |0       |1.0           |
|HAYAI                                            |148     |0       |1.0           |
|FIND ALL 2: Middle Ages                          |132     |0       |1.0           |
|Touhou Kaeizuka ～ Phantasmagoria of Flower View.|119     |0       |1.0           |
|Lucy Dreaming                                    |118     |0       |1.0

##### Are there years with more releases? Were there more or fewer game releases during the Covid, for example?

In [0]:
# Detect date format
df.select("data.release_date").distinct().show(3, truncate=False)


+------------+
|release_date|
+------------+
|2020/10/16  |
|2020/10/14  |
|2000/11/1   |
+------------+
only showing top 3 rows



In [0]:
from pyspark.sql.functions import to_date, year, col

# Authorize the date format
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

# Flatten the nested data
df_flat = df.select("data.*")

# Parse release_date using the correct format
df_with_date = df_flat.withColumn("release_date_parsed", to_date(col("release_date"), "yyyy/MM/dd"))

# Drop rows where the date could not be parsed (just in case)
df_valid_dates = df_with_date.filter(col("release_date_parsed").isNotNull())

# Extract the release year
df_with_year = df_valid_dates.withColumn("release_year", year(col("release_date_parsed")))

# Count releases per year
df_releases_by_year = df_with_year.groupBy("release_year").count().orderBy("release_year")


In [0]:
# Display a chart
df_releases_by_year.display()


Databricks visualization. Run in Databricks to view.

release_year,count
1997,2
1998,1
1999,3
2000,2
2001,4
2002,1
2003,3
2004,6
2005,6
2006,61


In [0]:
# Find the last release date

from pyspark.sql.functions import to_date, max

df_latest_date = df_with_date.select(to_date(col("release_date"), "yyyy/MM/dd").alias("release_date_parsed")) \
                             .filter(col("release_date_parsed").isNotNull()) \
                             .agg(max("release_date_parsed").alias("latest_release_date"))

df_latest_date.show()


+-------------------+
|latest_release_date|
+-------------------+
|         2022-11-11|
+-------------------+



The number of game releases on Steam experienced strong acceleration between 2013 and 2018. This was followed by a period of stabilization during 2019 and 2020, and then a renewed upward trend from 2021 onward. Although the data for 2022 is incomplete — with the latest release recorded on November 11th — the year can be reasonably estimated to reach around 9,000 releases, based on the 7,200 games published by that date.

##### How are the prizes distributed? Are there many games with a discount?

In [0]:
from pyspark.sql.functions import col, regexp_replace

# Flatten structure
df_flat = df.select("data.*")

# Filter out null or empty prices
df_prices = df_flat.filter((col("price").isNotNull()) & (col("price") != ""))

# Clean price string and convert to float (replace comma with dot if needed)
df_prices_clean = df_prices.withColumn("price_float", regexp_replace("price", ",", ".").cast("float"))

# Bucket prices into €5 ranges for aggregation
df_prices_binned = df_prices_clean.withColumn("price_bin", (col("price_float") / 5).cast("int") * 5)

# Group by price bin
df_price_distribution = df_prices_binned.groupBy("price_bin").count().orderBy("price_bin")
df_price_distribution.show()


+---------+-----+
|price_bin|count|
+---------+-----+
|        0| 7780|
|       25|   31|
|       30|    8|
|       35|   33|
|       40|   11|
|       45|  305|
|       50|  102|
|       55|  102|
|       60|    6|
|       65|   41|
|       70|   35|
|       75|   47|
|       80|    8|
|       85|   25|
|       90|   36|
|       95| 5245|
|      100|   26|
|      105|    6|
|      110|    1|
|      115|   40|
+---------+-----+
only showing top 20 rows



In [0]:
# Visualization of discount column

df.select("data.discount") \
  .groupBy("discount") \
  .count() \
  .orderBy(col("count").desc()) \
  .show(20, truncate=False)



+--------+-----+
|discount|count|
+--------+-----+
|0       |53173|
|50      |350  |
|90      |239  |
|80      |228  |
|75      |223  |
|40      |164  |
|51      |146  |
|70      |137  |
|60      |137  |
|30      |131  |
|20      |112  |
|25      |85   |
|10      |53   |
|15      |45   |
|35      |39   |
|65      |37   |
|87      |36   |
|85      |34   |
|83      |26   |
|48      |25   |
+--------+-----+
only showing top 20 rows



In [0]:
from pyspark.sql.functions import col

# Total number of games
total_games = df.select("data.*").count()

# Number of games with real discounts (discount != "0")
discounted_games = df.select("data.discount") \
    .filter((col("discount").isNotNull()) & (col("discount") != "") & (col("discount") != "0")) \
    .count()

# Display ratio
print(f"{discounted_games} out of {total_games} games have a real discount.")
print(f"{(discounted_games / total_games) * 100:.2f}% of games are discounted.")


2518 out of 55691 games have a real discount.
4.52% of games are discounted.


##### What are the most represented languages?

In [0]:
from pyspark.sql.functions import col, split, explode, trim

# Flatten the structure
df_flat = df.select("data.*")

# Filter non-null, non-empty language values
df_lang = df_flat.filter((col("languages").isNotNull()) & (col("languages") != ""))

# Split the language string and explode into individual rows
df_lang_split = df_lang.withColumn("language", explode(split(col("languages"), ",")))

# Clean whitespace (e.g. " French")
df_lang_trimmed = df_lang_split.withColumn("language", trim(col("language")))

# Group by language and count
df_language_counts = df_lang_trimmed.groupBy("language").count().orderBy(col("count").desc())


In [0]:
# Plot the data - TOP20 language

df_language_top20 = df_language_counts.limit(20)
display(df_language_top20)



Databricks visualization. Run in Databricks to view.

language,count
English,55116
German,14019
French,13426
Russian,12922
Simplified Chinese,12782
Spanish - Spain,12233
Japanese,10368
Italian,9304
Portuguese - Brazil,6750
Korean,6600


English overwhelmingly dominates as the primary language, reflecting its necessity for global reach. Other major languages include German, French, Russian, Chinese, Spanish, Japanese, and Italian.

##### Are there many games prohibited for children under 16/18?

In [0]:
from pyspark.sql.functions import col

# Flatten structure
df_flat = df.select("data.*")

# Convert required_age to integer
df_age = df_flat.withColumn("required_age_int", col("required_age").cast("int"))

# Count how many games require 16+ and 18+
df_age_stats = df_age.groupBy("required_age_int").count().orderBy("required_age_int")
df_age_stats.show()


+----------------+-----+
|required_age_int|count|
+----------------+-----+
|            null|    3|
|               0|55030|
|               3|    3|
|               5|    1|
|               6|    4|
|               7|    2|
|               8|    3|
|               9|    1|
|              10|    7|
|              12|   32|
|              13|   26|
|              14|   10|
|              15|  264|
|              16|   38|
|              17|   38|
|              18|  223|
|              20|    1|
|              35|    1|
|             180|    4|
+----------------+-----+



In [0]:
games_16plus = df_age.filter(col("required_age").cast("int") >= 16).count()
total_games = df_age.count()

print(f"{games_16plus} out of {total_games} games require age 16 or older.")
print(f"{(games_16plus / total_games) * 100:.2f}% of games are restricted to players aged 16 or above.")


305 out of 55691 games require age 16 or older.
0.55% of games are restricted to players aged 16 or above.


In [0]:
games_18plus = df_age.filter(col("required_age").cast("int") >= 18).count()
total_games = df_age.count()

print(f"{games_18plus} out of {total_games} games require age 18 or older.")
print(f"{(games_18plus / total_games) * 100:.2f}% of games are restricted to players aged 18 or above.")


229 out of 55691 games require age 18 or older.
0.41% of games are restricted to players aged 18 or above.


In [0]:
display(df_age_stats)

Databricks visualization. Run in Databricks to view.

required_age_int,count
null,3
0,55030
3,3
5,1
6,4
7,2
8,3
9,1
10,7
12,32


The vast majority of Steam games have no age restriction or a low minimum age requirement. However, a notable portion of the catalog is restricted to players aged 16 or 18 and above, indicating the presence of mature or adult content.

## 🎮 3. Genre-Based Analysis
Which genres are most common and which perform best?

##### What are the most represented genres?

In [0]:
from pyspark.sql.functions import col, split, explode, trim

# Flatten structure
df_flat = df.select("data.*")

# Filter non-empty genres
df_genre = df_flat.filter((col("genre").isNotNull()) & (col("genre") != ""))

# Split genre string into list (assumes comma-separated)
df_genre_split = df_genre.withColumn("genre_indiv", explode(split(col("genre"), ",")))

# Clean extra spaces
df_genre_trimmed = df_genre_split.withColumn("genre_indiv", trim(col("genre_indiv")))

# Count genres
df_genre_counts = df_genre_trimmed.groupBy("genre_indiv").count().orderBy(col("count").desc())

# Show top genres
df_genre_counts.show(20, truncate=False)


+---------------------+-----+
|genre_indiv          |count|
+---------------------+-----+
|Indie                |39681|
|Action               |23759|
|Casual               |22086|
|Adventure            |21431|
|Strategy             |10895|
|Simulation           |10836|
|RPG                  |9534 |
|Early Access         |6145 |
|Free to Play         |3393 |
|Sports               |2666 |
|Racing               |2155 |
|Massively Multiplayer|1460 |
|Utilities            |682  |
|Design & Illustration|406  |
|Animation & Modeling |322  |
|Education            |317  |
|Video Production     |247  |
|Audio Production     |195  |
|Violent              |168  |
|Software Training    |164  |
+---------------------+-----+
only showing top 20 rows



In [0]:
df_genre_top20 = df_genre_counts.limit(20)
display(df_genre_top20)

genre_indiv,count
Indie,39681
Action,23759
Casual,22086
Adventure,21431
Strategy,10895
Simulation,10836
RPG,9534
Early Access,6145
Free to Play,3393
Sports,2666


Databricks visualization. Run in Databricks to view.

"Indie" is by far the most represented genre on Steam, but it's a broad and somewhat ambiguous category — it simply refers to independently developed games and does not describe a specific gameplay style or theme. As a result, indie games can span any genre, from platformers to RPGs.
Among the more specific genres, Action, Casual, and Adventure are the most represented.
The term "Casual" typically refers to games that are easy to learn, accessible to a wide audience, and designed for short, relaxed play sessions — such as puzzle games, card games, or simple simulators.

##### Are there any genres that have a better positive/negative review ratio?

In [0]:
from pyspark.sql.functions import col, split, explode, trim

# Flatten and explode genres
df_flat = df.select("data.*")
df_genre = df_flat.filter((col("genre").isNotNull()) & (col("genre") != ""))
df_genre_split = df_genre.withColumn("genre_indiv", explode(split(col("genre"), ",")))
df_genre_trimmed = df_genre_split.withColumn("genre_indiv", trim(col("genre_indiv")))

# Filter out games with too few reviews
df_genre_filtered = df_genre_trimmed.filter((col("positive") + col("negative") >= 100))

# Compute positive ratio
df_with_ratio = df_genre_filtered.withColumn(
    "positive_ratio", col("positive") / (col("positive") + col("negative"))
)

# Aggregate: average ratio by genre
df_genre_ratio = df_with_ratio.groupBy("genre_indiv") \
    .avg("positive_ratio") \
    .withColumnRenamed("avg(positive_ratio)", "avg_positive_ratio") \
    .orderBy(col("avg_positive_ratio").desc())

# Show top 20 genres by rating ratio
df_genre_ratio.show(20, truncate=False)


+---------------------+------------------+
|genre_indiv          |avg_positive_ratio|
+---------------------+------------------+
|Game Development     |0.8257796145431594|
|Photo Editing        |0.8257165714669726|
|Web Publishing       |0.8191462394287768|
|Animation & Modeling |0.8103836893346357|
|Design & Illustration|0.8075536507255923|
|Sexual Content       |0.7976123117943439|
|Casual               |0.7970813855549078|
|Adventure            |0.7933820876944823|
|Indie                |0.7912639639445219|
|Utilities            |0.7830347853549947|
|Education            |0.7766577818501059|
|RPG                  |0.7748140155339491|
|Action               |0.7703330892964919|
|Nudity               |0.7667803236716946|
|Simulation           |0.7607264997787848|
|Software Training    |0.7601382989967299|
|Strategy             |0.7589381699262258|
|Sports               |0.7548743299612206|
|Racing               |0.7547281732445806|
|Video Production     |0.7547101987662453|
+----------

In [0]:
display(df_genre_ratio.limit(20))


genre_indiv,avg_positive_ratio
Game Development,0.8257796145431594
Photo Editing,0.8257165714669726
Web Publishing,0.8191462394287768
Animation & Modeling,0.8103836893346357
Design & Illustration,0.8075536507255923
Sexual Content,0.7976123117943439
Casual,0.7970813855549078
Adventure,0.7933820876944823
Indie,0.7912639639445219
Utilities,0.7830347853549947


Databricks visualization. Run in Databricks to view.

The average positive review ratios across genres are relatively similar, with no single category standing out significantly. This suggests that player satisfaction is fairly consistent across different types of games, regardless of genre.

##### Do some publishers have favorite genres?

In [0]:
from pyspark.sql.functions import col, split, explode, trim

# Flatten and explode genres
df_flat = df.select("data.*")
df_genre = df_flat.filter((col("genre").isNotNull()) & (col("genre") != ""))
df_genre_split = df_genre.withColumn("genre_indiv", explode(split(col("genre"), ",")))
df_genre_trimmed = df_genre_split.withColumn("genre_indiv", trim(col("genre_indiv")))

# Filter non-null publishers
df_publisher_genre = df_genre_trimmed.filter((col("publisher").isNotNull()) & (col("publisher") != ""))

# Group by publisher and genre
df_publisher_pref = df_publisher_genre.groupBy("publisher", "genre_indiv").count()

# Sort by publisher then descending count
df_publisher_pref_sorted = df_publisher_pref.orderBy("publisher", col("count").desc())

# Show example: top 20 publisher-genre pairs
df_publisher_pref_sorted.show(20, truncate=False)


+-----------------------------+------------+-----+
|publisher                    |genre_indiv |count|
+-----------------------------+------------+-----+
|                             |Indie       |2    |
|                             |Action      |2    |
|                             |Casual      |1    |
| AK Studio                   |Indie       |2    |
| AK Studio                   |Adventure   |1    |
| AK Studio                   |Action      |1    |
| AK Studio                   |Casual      |1    |
| AK Studio                   |Racing      |1    |
| ARVORE Immersive Experiences|Adventure   |1    |
| ARVORE Immersive Experiences|Casual      |1    |
| ARVORE Immersive Experiences|Strategy    |1    |
| ARVORE Immersive Experiences|Action      |1    |
| ARVORE Immersive Experiences|Indie       |1    |
| Alon Zubina                 |Indie       |1    |
| Alon Zubina                 |Simulation  |1    |
| Alon Zubina                 |Adventure   |1    |
| Appnori Inc.                |

In [0]:
from pyspark.sql.functions import col

top_publishers = df_flat.filter((col("publisher").isNotNull()) & (col("publisher") != "")) \
    .groupBy("publisher") \
    .count() \
    .orderBy(col("count").desc()) \
    .limit(20) \
    .select("publisher")

top_genres = df_genre_trimmed.groupBy("genre_indiv") \
    .count() \
    .orderBy(col("count").desc()) \
    .limit(20) \
    .select("genre_indiv")

# Jointure sur les 20 publishers et genres
df_heatmap_base = df_publisher_genre \
    .join(top_publishers, on="publisher") \
    .join(top_genres, on="genre_indiv")

# Compter le nombre de jeux par (éditeur, genre)
df_heatmap = df_heatmap_base.groupBy("publisher", "genre_indiv").count()



In [0]:
from pyspark.sql.functions import col

df_heatmap_fixed = df_heatmap.select(
    col("publisher"),
    col("genre_indiv"),
    col("count").cast("int").alias("count")
)


In [0]:

display(df_heatmap_fixed)

publisher,genre_indiv,count
Slitherine Ltd.,Indie,9
Big Fish Games,Action,1
HH-Games,Indie,69
Ubisoft,Racing,14
8floor,Strategy,22
HH-Games,Adventure,39
Choice of Games,Casual,28
Big Fish Games,Casual,418
Ubisoft,Free to Play,6
Plug In Digital,Audio Production,1


Databricks visualization. Run in Databricks to view.

This heatmap displays the distribution of games published by the top 20 publishers across the 20 most common genres. It reveals strong preferences from some publishers for specific genres, while others maintain a more diversified portfolio.

##### What are the most lucrative genres?

In [0]:
from pyspark.sql.functions import col, split, explode, trim, regexp_replace

# Étape 1 : Flatten et exploser les genres
df_flat = df.select("data.*")
df_genre = df_flat.filter((col("genre").isNotNull()) & (col("genre") != ""))
df_genre_split = df_genre.withColumn("genre_indiv", explode(split(col("genre"), ",")))
df_genre_trimmed = df_genre_split.withColumn("genre_indiv", trim(col("genre_indiv")))

# Étape 2 : Nettoyer le prix (string -> float)
df_with_price = df_genre_trimmed \
    .filter((col("price").isNotNull()) & (col("price") != "")) \
    .withColumn("price_float", regexp_replace("price", ",", ".").cast("float"))

# Étape 3 : Estimer le revenu par jeu
df_with_revenue = df_with_price.withColumn(
    "estimated_revenue", col("price_float") * (col("positive") + col("negative"))
)

# Étape 4 : Agréger par genre
df_genre_revenue = df_with_revenue.groupBy("genre_indiv") \
    .sum("estimated_revenue") \
    .withColumnRenamed("sum(estimated_revenue)", "total_estimated_revenue") \
    .orderBy(col("total_estimated_revenue").desc())

# Afficher top 20
df_genre_revenue.show(20, truncate=False)

+---------------------+-----------------------+
|genre_indiv          |total_estimated_revenue|
+---------------------+-----------------------+
|Action               |1.05306097077E11       |
|Adventure            |6.7822663179E10        |
|Indie                |5.0648939176E10        |
|RPG                  |4.9239517409E10        |
|Simulation           |3.4066895454E10        |
|Strategy             |3.0288584761E10        |
|Massively Multiplayer|1.1609725614E10        |
|Casual               |1.0335681212E10        |
|Early Access         |8.751852694E9          |
|Sports               |5.755989802E9          |
|Racing               |5.47908997E9           |
|Design & Illustration|4.43423181E8           |
|Utilities            |4.38251851E8           |
|Animation & Modeling |3.63198881E8           |
|Photo Editing        |2.85662993E8           |
|Audio Production     |1.3777156E8            |
|Video Production     |1.27914885E8           |
|Web Publishing       |1.22858029E8     

In [0]:
display(df_genre_revenue.limit(20))

Databricks visualization. Run in Databricks to view.

genre_indiv,total_estimated_revenue
Action,1.05306097077E11
Adventure,6.7822663179E10
Indie,5.0648939176E10
RPG,4.9239517409E10
Simulation,3.4066895454E10
Strategy,3.0288584761E10
Massively Multiplayer,1.1609725614E10
Casual,1.0335681212E10
Early Access,8.751852694E9
Sports,5.755989802E9


Based on an estimated revenue calculation (price multiplied by the number of reviews), action genre appear to be the most lucrative. This highlights the commercial weight of high-engagement, content-rich genres.

## 🖥️ 4. Platform Distribution
Are games mostly available on Windows, Mac, or Linux?

##### Are most games available on Windows/Mac/Linux instead?

In [0]:
from pyspark.sql.functions import col

# Flatten the structure
df_flat = df.select("data.*")

# Extract platform availability
df_platforms = df_flat.select(
    col("name"),
    col("platforms.windows").alias("windows"),
    col("platforms.mac").alias("mac"),
    col("platforms.linux").alias("linux")
)

# Count how many games are available per platform
df_platform_counts = df_platforms.select("windows", "mac", "linux") \
    .groupBy("windows", "mac", "linux") \
    .count() \
    .orderBy(col("count").desc())

df_platform_counts.show(truncate=False)


+-------+-----+-----+-----+
|windows|mac  |linux|count|
+-------+-----+-----+-----+
|true   |false|false|41271|
|true   |true |true |6807 |
|true   |true |false|5951 |
|true   |false|true |1647 |
|false  |true |false|11   |
|false  |false|true |3    |
|false  |true |true |1    |
+-------+-----+-----+-----+



In [0]:
df_platform_totals = df_platforms.select(
    col("windows").cast("int"),
    col("mac").cast("int"),
    col("linux").cast("int")
).agg(
    {"windows": "sum", "mac": "sum", "linux": "sum"}
)

df_platform_totals.show()


+----------+------------+--------+
|sum(linux)|sum(windows)|sum(mac)|
+----------+------------+--------+
|      8458|       55676|   12770|
+----------+------------+--------+



##### Do certain genres tend to be preferentially available on certain platforms?

In [0]:
from pyspark.sql.functions import col, split, explode, trim

# Flatten and explode genres
df_flat = df.select("data.*")

df_genre = df_flat.filter((col("genre").isNotNull()) & (col("genre") != ""))

df_genre_split = df_genre.withColumn("genre_indiv", explode(split(col("genre"), ",")))

df_genre_trimmed = df_genre_split.withColumn("genre_indiv", trim(col("genre_indiv")))

# Extraire plateformes associées à chaque ligne de genre
df_genre_platform = df_genre_trimmed.select(
    "genre_indiv",
    col("platforms.windows").alias("windows"),
    col("platforms.mac").alias("mac"),
    col("platforms.linux").alias("linux")
)


In [0]:
from pyspark.sql.functions import col

# Convert booleans to integers
df_genre_platform_int = df_genre_platform.select(
    col("genre_indiv"),
    col("windows").cast("int").alias("windows_int"),
    col("mac").cast("int").alias("mac_int"),
    col("linux").cast("int").alias("linux_int")
)

# Aggregate availability counts per genre
df_platform_stats = df_genre_platform_int.groupBy("genre_indiv").agg(
    {"windows_int": "sum", "mac_int": "sum", "linux_int": "sum"}
).withColumnRenamed("sum(windows_int)", "windows_count") \
 .withColumnRenamed("sum(mac_int)", "mac_count") \
 .withColumnRenamed("sum(linux_int)", "linux_count")

# Optional: sort
df_platform_stats_sorted = df_platform_stats.orderBy(col("windows_count").desc())

# Display
df_platform_stats_sorted.show(20, truncate=False)



+---------------------+-------------+---------+-----------+
|genre_indiv          |windows_count|mac_count|linux_count|
+---------------------+-------------+---------+-----------+
|Indie                |39676        |9935     |6978       |
|Action               |23755        |4564     |3379       |
|Casual               |22082        |5130     |3305       |
|Adventure            |21427        |5039     |3302       |
|Strategy             |10892        |3005     |1826       |
|Simulation           |10832        |2439     |1532       |
|RPG                  |9533         |2248     |1524       |
|Early Access         |6145         |900      |632        |
|Free to Play         |3391         |845      |474        |
|Sports               |2665         |506      |287        |
|Racing               |2154         |424      |304        |
|Massively Multiplayer|1459         |270      |164        |
|Utilities            |681          |102      |49         |
|Design & Illustration|405          |100

In [0]:
# Limiter aux 20 genres les plus répandus
df_platform_stats_top20 = df_platform_stats_sorted.limit(20)
display(df_platform_stats_top20)

genre_indiv,windows_count,mac_count,linux_count
Indie,39676,9935,6978
Action,23755,4564,3379
Casual,22082,5130,3305
Adventure,21427,5039,3302
Strategy,10892,3005,1826
Simulation,10832,2439,1532
RPG,9533,2248,1524
Early Access,6145,900,632
Free to Play,3391,845,474
Sports,2665,506,287


Databricks visualization. Run in Databricks to view.

The platform distribution appears fairly consistent across genres, with the vast majority of games available on Windows, while Mac and Linux support remains limited and secondary.

%md
## 🔍 5. Additional Analysis  
In this final section, we explore complementary aspects of the dataset to enrich our understanding of the Steam ecosystem. This includes cross-dimensional insights such as genre-platform relationships, revenue estimations, and publisher preferences. These additional perspectives help refine strategic thinking for game design, marketing, and platform targeting.


####### 5.1 Monthly Game Release Trends  
We examine the temporal distribution of game releases to detect seasonal or structural patterns in Steam’s publishing activity.


In [0]:
from pyspark.sql.functions import to_date, date_format, col

# Parser la date (format: yyyy/MM/dd)
df_with_date = df.select("data.*") \
    .withColumn("release_date_parsed", to_date(col("release_date"), "yyyy/MM/dd"))

# Extraire l’année et le mois au format "YYYY-MM"
df_with_month = df_with_date \
    .filter(col("release_date_parsed").isNotNull()) \
    .withColumn("release_month", date_format(col("release_date_parsed"), "yyyy-MM"))


In [0]:
df_releases_by_month = df_with_month.groupBy("release_month") \
    .count() \
    .orderBy("release_month")

display(df_releases_by_month)

Databricks visualization. Run in Databricks to view.

release_month,count
1997-06,1
1997-11,1
1998-11,1
1999-04,1
1999-09,1
1999-11,1
2000-11,2
2001-03,1
2001-06,2
2001-12,1


Since 2017 (with the exception of 2020), the months with the highest number of game releases have consistently been March and October. March is often targeted by publishers aiming to meet fiscal year-end goals, as many major gaming companies close their financial year on March 31st. October releases are typically positioned ahead of the holiday season to maximize visibility and sales during the peak shopping months.

###### 5.2 Review Volume vs. Satisfaction Score  
This section explores the relationship between a game's popularity (number of reviews) and its approval rating to identify potential trade-offs between reach and satisfaction.

In [0]:
from pyspark.sql.functions import col

# Flatten and filter valid review data
df_flat = df.select("data.*")

df_reviews = df_flat.filter(
    (col("positive").isNotNull()) & 
    (col("negative").isNotNull()) & 
    ((col("positive") + col("negative")) >= 10)  # filtre pour éviter les jeux avec trop peu de votes
)

# Ajouter total votes et ratio positif
df_reviews_ratio = df_reviews.withColumn("total_votes", col("positive") + col("negative")) \
    .withColumn("positive_ratio", col("positive") / col("total_votes"))


In [0]:
df_reviews_filtered = df_reviews_ratio.filter(col("total_votes") > 20000)
display(df_reviews_filtered.select("total_votes", "positive_ratio"))


Databricks visualization. Run in Databricks to view.

total_votes,positive_ratio
206414,0.9748127549487923
42774,0.9183382428578108
48682,0.9320488065404051
48120,0.8672069825436409
89313,0.8246279936851298
28967,0.751406773224704
43710,0.7955387783115991
29144,0.909037880867417
30725,0.9076973148901546
1037091,0.978420408623737


The analysis reveals that the most popular games — those with the highest number of reviews — are not necessarily the best rated. While some maintain high satisfaction scores, others show more mixed feedback, suggesting that broader reach often comes with more diverse opinions.

###### 5.3 Top Publishers by Estimated Revenue  
We rank publishers based on estimated game revenue to highlight the most commercially dominant players on the platform.
Souhaites-tu commencer par l’implémentation de la première section (5.1 Monthly Game Release Trends) ?

In [0]:
from pyspark.sql.functions import col, regexp_replace

# Flatten data
df_flat = df.select("data.*")

# Nettoyer et convertir le prix (virgule -> point, puis float)
df_price_clean = df_flat.filter((col("price").isNotNull()) & (col("price") != "")) \
    .withColumn("price_float", regexp_replace("price", ",", ".").cast("float"))

# Filtrer les votes valides
df_price_votes = df_price_clean.filter(
    (col("positive").isNotNull()) & (col("negative").isNotNull())
)

# Estimer le revenu : price × (positive + negative)
df_revenue = df_price_votes.withColumn(
    "estimated_revenue", col("price_float") * (col("positive") + col("negative"))
)


In [0]:
df_publisher_revenue = df_revenue.filter((col("publisher").isNotNull()) & (col("publisher") != "")) \
    .groupBy("publisher") \
    .sum("estimated_revenue") \
    .withColumnRenamed("sum(estimated_revenue)", "total_estimated_revenue") \
    .orderBy(col("total_estimated_revenue").desc())


In [0]:
df_top_publishers_revenue = df_publisher_revenue.limit(30)
display(df_top_publishers_revenue)


Databricks visualization. Run in Databricks to view.

publisher,total_estimated_revenue
Ubisoft,7.485177194E9
Rockstar Games,6.705516782E9
Electronic Arts,6.512846459E9
CD PROJEKT RED,6.006666817E9
Xbox Game Studios,5.450974905E9
Bethesda Softworks,5.351278498E9
Paradox Interactive,3.774818165E9
Facepunch Studios,3.379137855E9
"FromSoftware Inc., Bandai Namco Entertainment",3.253083648E9
Valve,3.170698085E9


The largest publishers can achieve extremely high revenues — with the leader, Ubisoft, surpassing $7 billion in estimated earnings. However, only about thirty publishers exceed the $1 billion mark, while the vast majority generate significantly lower revenues. This highlights a stark disparity in financial capacity and market power across publishers on Steam.

## 🧠 6. Conclusions and Insights
We summarize the key findings from our analysis, highlighting the most influential factors affecting game popularity, the most active publishers and genres, and any platform-specific trends. These insights can help inform strategic decisions for future game development and publishing on Steam.


Our comprehensive analysis of the Steam game catalog reveals several strategic insights for publishers looking to succeed on the platform. First, game popularity and estimated revenue are not solely driven by volume of reviews or presence in dominant genres — although Action, Indie, and Adventure titles remain highly represented, they do not guarantee high satisfaction or earnings. The most lucrative genres combine both wide appeal and higher price points, often supported by strong engagement (as seen in RPG and Strategy). However, popularity does not always correlate with high review scores, indicating the importance of audience alignment and expectation management.

Platform-wise, Windows remains the de facto standard, while Mac and Linux support is secondary and genre-dependent. Release timing also plays a significant role: March and October consistently emerge as key launch windows, likely influenced by industry cycles and Steam sales. Finally, the analysis reveals a sharp disparity in revenue across publishers — only a handful dominate the market with billion-dollar portfolios, while the majority operate with limited financial impact. This imbalance underscores the importance of strategic positioning, quality execution, and market differentiation for smaller studios aiming to stand out.

Overall, successful publishing on Steam depends not only on choosing the right genre or timing but also on delivering consistent value to a targeted audience, leveraging platform trends, and navigating a highly competitive and uneven landscape.